# AI 부트캠프 Day 1 — PyTorch 기초와 어텐션의 탄생

> **이 부트캠프의 목표: PyTorch로 텐서를 자유롭게 다루는 법을 익히는 것**
>
> 텐서를 만들고, 모양을 바꾸고, 곱하고, 원하는 방향(dim)으로 연산하는 것에 익숙해지면
> 어떤 최신 모델도 뜯어보고 직접 만들 수 있습니다. 이번 실습에서는 2일에 걸쳐
> ChatGPT의 핵심 구조인 **트랜스포머 인코더**를 처음부터 직접 구현해 봅니다.

## 2일 일정표

| 시간 | 내용 |
|---|---|
| **Day 1 오전** | 오리엔테이션: 서버/주피터/커널/GPU 세팅 → 왜 PyTorch인가 → 텐서 생성 기초 실습 → 트랜스포머를 왜 배우나 |
| **Day 1 오후** | 토큰 → 임베딩 → 어텐션 계산(1:1 → 1:다) → Self-Attention |
| **Day 2 오전** | Multi-Head Attention |
| **Day 2 오후** | FeedForward · LayerNorm · Residual → 인코더 1-layer → 인코더 완성(layer 쌓기) |

- 이 수업의 핵심은 PyTorch를 직접 다뤄보며 익히는 것입니다. 눈으로만 보지 말고 **직접 타이핑**하면서 진행하세요.

---
# Part 1. [오전-오리엔테이션] 서버 접속 & 주피터 열기

> 딥러닝 실습은 GPU가 있는 서버에서 진행합니다.
> 각자에게 **NVIDIA A6000 GPU 1개**가 배정되어 있습니다.
>  내 주피터 노트북이 해당 GPU와 정상적으로 연결되어 있는지 확인해봅니다.

1. 브라우저를 열고 아래 주소로 접속
   - 접속 주소: `http://<서버주소>:<포트>`
   - 로그인 계정/비밀번호: 현장에서 안내
2. 접속하면 JupyterLab(또는 Jupyter Notebook) 화면이 보임
3. 왼쪽 파일 탐색기에서 이 파일을 더블클릭해서 열기
4. 셀 실행 방법: 셀을 클릭하고 `Shift + Enter`

# Part 2. 주피터에 conda 환경(커널) 연결하기

서버에는 여러 파이썬 환경이 설치되어 있음. 실습에서는 **부트캠프 전용 환경**을 사용해야 함  

1. 노트북 화면 오른쪽 위의 커널 이름을 클릭하거나, 메뉴에서 **Kernel → Change Kernel...** 을 선택
2. 목록에서 안내받은 커널 이름(예: `bootcamp`)을 선택
3. 아래 셀을 실행해서 현재 어떤 파이썬을 쓰고 있는지 확인

In [ ]:
# 지금 이 노트북이 사용 중인 파이썬(커널)의 위치를 확인합니다.
# 출력에 부트캠프 환경 이름(예: .../envs/bootcamp/...)이 보이면 됩니다.
import sys

print("파이썬 실행 경로:", sys.executable)
print("파이썬 버전   :", sys.version)

# Part 3. 필수 라이브러리 설치/확인

실습에 필요한 라이브러리는 이미 서버에 설치되어 있음  
아래 셀은 설치가 아니라 버전 확인용. 에러 없이 버전이 출력되면 됩니다.

In [ ]:
# 필수 라이브러리 버전 확인 (에러 없이 4줄이 출력되면 OK)
import numpy
import pandas
import matplotlib
import torch



# Part 4. GPU 인식 확인

> **GPU가 왜 필요한가요?** 딥러닝의 계산은 대부분 "아주 큰 행렬 곱셈".  
> GPU는 수천 개의 코어로 이런 계산을 동시에 처리해서 CPU보다 수십~수백 배 빠름

아래 셀을 실행해서 `True`와 GPU 이름(`NVIDIA RTX A6000`)이 나오는지 확인

In [ ]:
# GPU 인식 확인
import torch

print("CUDA(GPU) 사용 가능 여부:", torch.cuda.is_available())



## 트러블슈팅 — 자주 나는 문제들

| 증상 | 원인/해결 |
|---|---|
| **커널 목록에 부트캠프 환경이 안 보임** | 브라우저 새로고침(F5) → Kernel → Change Kernel 다시 시도. 그래도 없으면 문의하기 |
| **`ModuleNotFoundError: No module named 'torch'`** | 커널을 잘못 선택한 경우가 90%. Part 2로 돌아가 커널 다시 선택 |
| **`torch.cuda.is_available()`이 `False`** | ① 커널 선택 확인 → ② 메뉴 Kernel → Restart Kernel 후 재실행 → ③ 그래도 안 되면 GPU 배정 문제일 수 있으니 문의하기 |
| **`CUDA error: out of memory`** | 다른 프로세스(예: 이전에 켜 둔 다른 노트북 커널)가 GPU 메모리를 점유 중. 안 쓰는 노트북의 커널을 종료(Kernel → Shut Down Kernel)한 뒤 재시도 |
| **셀이 `[*]` 표시로 멈춰 있음** | 이전 셀이 아직 실행 중. 잠시 대기 → 너무 길면 Kernel → Interrupt |
| **`NameError: name '___' is not defined`** | 빈칸(`___`)을 아직 채우지 않고 실행한 것. 라이브 코딩을 따라 빈칸을 채운 뒤 실행 |

---
# Part 5. PyTorch를 왜 사용하는가? 왜 좋은가?

> 앞으로 2일간 사용할 도구가 PyTorch. 뭐가 좋아서 전 세계 표준이 됐을까?

**PyTorch = NumPy + 두 가지 강점**

| | NumPy | PyTorch |
|---|---|---|
| 다차원 배열 계산 | ✅ `ndarray` | ✅ `Tensor` (사용법이 거의 같음) |
| **① GPU 가속** | ❌ | ✅ `.to("cuda")` 한 줄이면 GPU에서 계산 |
| **② 자동 미분** | ❌ (미분을 손으로 계산) | ✅ `.backward()` 한 줄이면 기울기 자동 계산 |

딥러닝은 "미분으로 모델을 조금씩 개선하는 것"의 반복이므로,  
**GPU(속도)** 와 **자동미분(학습)** 이 기본으로 제공되는 PyTorch가 표준 도구가 됨

말로만 하면 이해가 어려우니, 두 강점을 지금 직접 확인 (아래 두 셀은 실행)

In [5]:
# 증거 ① GPU 가속: (4096 × 4096) 행렬 두 개를 곱하는 시간을 CPU vs GPU로 비교
import time
import torch



matrix_size = 4096
matrix_a = torch.rand(matrix_size, matrix_size)
matrix_b = torch.rand(matrix_size, matrix_size)

# --- CPU에서 행렬곱 ---
start_time = time.time()


cpu_seconds = time.time() - start_time
print(f"CPU 소요 시간: {cpu_seconds:.4f}초")



CPU 소요 시간: 0.0010초


In [6]:
# --- GPU에서 행렬곱 ---
if torch.cuda.is_available():


    torch.cuda.synchronize()              # GPU 작업이 끝날 때까지 대기 (정확한 시간 측정용)

    start_time = time.time()

    
    torch.cuda.synchronize()
    gpu_seconds = time.time() - start_time
    print(f"GPU 소요 시간: {gpu_seconds:.4f}초")
    print(f"→ GPU가 약 {cpu_seconds / gpu_seconds:.0f}배 빠릅니다!")

In [ ]:
# 증거 ② 자동 미분: y = x² + 3x 를 x=2에서 미분하면? (손으로 하면 2x+3 → 7)
x = torch.tensor(2.0, requires_grad=True)   # "이 변수로 미분할 거야"라고 표시
y = x ** 2 + 3 * x                          # 계산식을 만들고
y.backward()                                # 미분 실행
print("PyTorch가 구한 기울기 dy/dx:", x.grad)   # tensor(7.) ← 손 미분과 정확히 일치

# 아무리 복잡한 식이라도(수억 개의 파라미터라도) 이렇게 자동으로 미분됩니다.
# 딥러닝 "학습" = 이 기울기의 반대 방향으로 파라미터를 조금씩 수정하는 것의 반복입니다.

PyTorch가 구한 기울기 dy/dx: tensor(7.)


---
# Part 6. 텐서 생성 기초

> **왜 배우나요?** 문장도, 이미지도, 모델의 파라미터도 전부 **Tensor**.  
> PyTorch를 다룬다 = 텐서를 다룬다. 자유자재로 만들 수 있어야 시작

**Tensor = 숫자를 담는 다차원 상자**

- 0차원: 스칼라 `3.14`
- 1차원: 벡터 `[1, 2, 3]`
- 2차원: 행렬 `[[1, 2], [3, 4]]`
- 3차원 이상: 그 이상의 상자 (예: 컬러 이미지 = 채널×높이×너비)

모든 Tensor는 3가지 속성을 가짐. 앞으로 이 셋을 계속 확인하게 됨

| 속성 | 의미 | 예 |
|---|---|---|
| `.shape` | 상자의 크기 (각 차원의 길이) | `torch.Size([2, 3])` |
| `.dtype` | 담긴 숫자의 종류 | `torch.float32`, `torch.int64` |
| `.device` | 상자가 놓인 장소 | `cpu` 또는 `cuda:0` |

### [질문] 리스트 `[[1, 2, 3], [4, 5, 6]]`을 Tensor로 만들면 shape은?

### [실습] Tensor 생성 5종 세트 + GPU로 보내기

| 함수 | 하는 일 |
|---|---|
| `torch.tensor(리스트)` | 리스트/숫자 → Tensor |
| `torch.zeros(모양)` | 0으로 가득 찬 Tensor |
| `torch.ones(모양)` | 1로 가득 찬 Tensor |
| `torch.rand(모양)` | 0~1 사이 랜덤 값 Tensor |
| `torch.arange(시작, 끝, 간격)` | 연속된 정수 Tensor |
| `.to(device)` | Tensor를 GPU(또는 CPU)로 이동 |

In [ ]:
import torch

# ① 리스트 [[1, 2, 3], [4, 5, 6]] 을 Tensor로
my_tensor = torch.tensor([[1, 2, 3], [4, 5, 6]])
print("my_tensor:\n", my_tensor)
print("  shape :", my_tensor.shape)    # 예상: torch.Size([2, 3])
print("  dtype :", my_tensor.dtype)    # 예상: torch.int64 (정수 리스트이기 때문)
print("  device:", my_tensor.device)   # 예상: cpu


In [ ]:
# ② 0으로 채워진 (2, 3) Tensor


In [ ]:
# ③ 1로 채워진 (3,) Tensor를 float32 타입으로


In [ ]:
# ④ 0~1 랜덤 값으로 채워진 (2, 4) Tensor


In [ ]:
# ⑤ 0부터 9까지 2씩 건너뛰는 Tensor → [0, 2, 4, 6, 8]


In [ ]:
# ⑥ random_tensor를 GPU로 보내기 (Part 5에서 만든 device 사용)


> 📌 **체크포인트**
> - 정수 리스트로 만들면 `dtype=torch.int64`, 소수점이 있으면 `torch.float32`
>   딥러닝 계산은 대부분 float32로 처리
> - `shape`을 읽는 법: `torch.Size([2, 3])` = "2행 3열" = 바깥 상자에 2개, 그 안에 각각 3개
> - 서로 다른 device의 Tensor끼리는 연산 불가 (`cpu` 텐서 + `cuda` 텐서 = 에러) — 가장 흔한 실수
>
> **[실습]** 숫자를 마음대로 바꿔보기. `torch.zeros(2, 3, 4)`처럼 3차원 상자도 만들어보고,
> `torch.arange(10, 0, -2)`처럼 거꾸로도 시도. `gpu_tensor + random_tensor`를 실행해서
> device가 다르면 어떤 에러가 나는지도 직접 확인

---
# Part 7. 트랜스포머는 왜 좋은가? 왜 배우는가?

트랜스포머 이전에는 **RNN(순환 신경망)** 이 문장 처리의 표준이었음  
RNN은 문장을 한 단어씩 순서대로 읽음. 사람이 책을 읽는 방식과 비슷함. 여기엔 치명적인 약점이 있음

| RNN의 약점 | 왜 문제인가 |
|---|---|
| **느림** | 단어를 하나 처리해야 다음 단어를 처리할 수 있음 → 아무리 GPU가 좋아도 병렬 처리 불가 |
| **까먹음** | 문장이 길어지면 앞부분 정보가 점점 희미해짐 (장기 의존성 문제). "그 영화 정말 재미없... (500단어) ...지 않았어!" 에서 결론을 놓침 |

2017년, "Attention Is All You Need"라는 논문 등장  

> 순서대로 읽지 말고, 모든 단어가 모든 단어를 동시에 참조하게 하자

- 모든 단어를 동시에 처리 → GPU 병렬 계산에 적합 (Part 5에서 본 그 속도)
- 아무리 먼 단어도 직접 연결 → 정보 손실 없음

이 "서로 참조하는" 메커니즘이 바로 **어텐션(Attention)** 이고, 오늘 오후에 직접 구현할 대상  
어텐션의 정체는 사실 오전에 배운 텐서 조작 몇 개의 조합. 텐서를 다룰 줄 알면 만들 수 있음
